In [1]:
import torchvision.models as models  # pyright: ignore[reportMissingTypeStubs]
import torch.nn as nn
from typing import Dict, Any

from model_ranking import (
    ClassificationModelConfig
)

INFO: P [MainThread] 2025-09-09 10:52:33,165 plantseg - Logger configured at initialisation. PlantSeg logger name: plantseg


/g/kreshuk/talks/miniforge3/envs/model-rank-local2/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/g/kreshuk/talks/pytorch-3dunet/pytorch3dunet/unet3d/predictor.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/g/kreshuk/talks/pytorch-3dunet/pytorch3dunet/unet3d/predictor.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [ ]:
model = models.resnet50()
print(model)

In [ ]:
model = models.resnet18()
print(model)

In [ ]:
model = models.densenet121()
print(model)

In [ ]:
model = models.densenet169()
print(model)

In [ ]:
model = models.mobilenet_v2()
print(model)

In [ ]:
model = models.mobilenet_v3_small()
print(model)

In [ ]:
model = models.vgg16_bn()
print(model)

In [ ]:
model = models.vgg19_bn()
print(model)

In [2]:
model_config: Dict[str, Any] = {
    "conv1": {"name": "ResNet18"},
    "out_channels": 1,
    "modelname": "RN18_epfl80_aug5",
    "ckpt_path": "/g/kreshuk/talks/FineTuning/experiments/patch_classification/epfl/checkpoints/",
    "ckpt_key": "model_state",
    "feature_layers": [7,8]
}

model_cfg = ClassificationModelConfig.model_validate(model_config)

In [3]:
conv1_cfg = model_cfg.conv1
input_conv = nn.Conv2d(
    conv1_cfg.in_channels,
    conv1_cfg.out_channels,
    kernel_size=conv1_cfg.kernel_size,
    stride=conv1_cfg.stride,
    padding=conv1_cfg.padding,
    bias=conv1_cfg.bias,
)

if model_cfg.conv1.name == "ResNet18":
    model = models.resnet18()
    model.conv1 = input_conv
    nr_filters = model.fc.in_features
    model.fc = nn.Linear(nr_filters, model_cfg.out_channels)
elif model_cfg.conv1.name == "ResNet50":
    model = models.resnet50()
    model.conv1 = input_conv
    nr_filters = model.fc.in_features
    model.fc = nn.Linear(nr_filters, model_cfg.out_channels)
elif model_cfg.conv1.name == "DenseNet121":
    model = models.densenet121()
    model.features.conv0 = input_conv
    nr_filters = model.classifier.in_features
    model.classifier = nn.Linear(nr_filters, model_cfg.out_channels)
elif model_cfg.conv1.name == "DenseNet169":
    model = models.densenet169()
    model.features.conv0 = input_conv
    nr_filters = model.classifier.in_features
    model.classifier = nn.Linear(nr_filters, model_cfg.out_channels)
elif model_cfg.conv1.name == "MobileNetV2":
    model = models.mobilenet_v2()
    model.features[0][0] = input_conv
    nr_filters = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(nr_filters, model_cfg.out_channels)
elif model_cfg.conv1.name == "MobileNetV3":
    model = models.mobilenet_v3_small()
    model.features[0][0] = input_conv
    nr_filters = model.classifier[3].in_features
    model.classifier[3] = nn.Linear(nr_filters, model_cfg.out_channels)
elif model_cfg.conv1.name == "VGG16":
    model = models.vgg16_bn()
    model.features[0] = input_conv
    nr_filters = model.classifier[6].in_features
    model.classifier[6] = nn.Linear(nr_filters, model_cfg.out_channels)
elif model_cfg.conv1.name == "VGG19":
    model = models.vgg19_bn()
    model.features[0] = input_conv
    nr_filters = model.classifier[6].in_features
    model.classifier[6] = nn.Linear(nr_filters, model_cfg.out_channels)
else:
    raise ValueError(f"Unknown model type: {model_cfg.conv1.name}")


print(model)

ResNet(
  (conv1): Conv2d(1, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  